In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/titanic/train.csv
/kaggle/input/titanic/test.csv
/kaggle/input/titanic/gender_submission.csv


In [69]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn import svm


In [82]:
data =pd.read_csv("/kaggle/input/titanic/train.csv")
# print(data)
data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [83]:
data['Embarked'].fillna('S', inplace=True)
data.loc[data.Fare.isnull(),'Fare'] = data['Fare'].mean()
data.loc[data.Age.isnull(),'Age'] = data['Age'].mean()

labels = data["Survived"]
features = data.drop(columns=['PassengerId','Survived','PassengerId','Cabin','Ticket','Name'])

hot =pd.get_dummies(features,columns=['Sex','Embarked'])
features =hot
features,labels

(     Pclass        Age  SibSp  Parch     Fare  Sex_female  Sex_male  \
 0         3  22.000000      1      0   7.2500           0         1   
 1         1  38.000000      1      0  71.2833           1         0   
 2         3  26.000000      0      0   7.9250           1         0   
 3         1  35.000000      1      0  53.1000           1         0   
 4         3  35.000000      0      0   8.0500           0         1   
 ..      ...        ...    ...    ...      ...         ...       ...   
 886       2  27.000000      0      0  13.0000           0         1   
 887       1  19.000000      0      0  30.0000           1         0   
 888       3  29.699118      1      2  23.4500           1         0   
 889       1  26.000000      0      0  30.0000           0         1   
 890       3  32.000000      0      0   7.7500           0         1   
 
      Embarked_C  Embarked_Q  Embarked_S  
 0             0           0           1  
 1             1           0           0  
 2   

In [71]:
X_train,X_test,y_train,y_test =train_test_split(features, labels, test_size=0.3)

In [60]:
X_train,y_train

(     Pclass   Age  SibSp  Parch      Fare  Sex_female  Sex_male  Embarked_C  \
 683       3  14.0      5      2   46.9000           0         1           0   
 320       3  22.0      0      0    7.2500           0         1           0   
 321       3  27.0      0      0    7.8958           0         1           0   
 450       2  36.0      1      2   27.7500           0         1           0   
 187       1  45.0      0      0   26.5500           0         1           0   
 ..      ...   ...    ...    ...       ...         ...       ...         ...   
 227       3  20.5      0      0    7.2500           0         1           0   
 418       2  30.0      0      0   13.0000           0         1           0   
 195       1  58.0      0      0  146.5208           1         0           1   
 658       2  23.0      0      0   13.0000           0         1           0   
 230       1  35.0      1      0   83.4750           1         0           0   
 
      Embarked_Q  Embarked_S  
 683   

In [87]:
def evaluate(model,y_test=y_test):
    predictions = model.predict(X_test)
    acc = accuracy_score(predictions,y_test)
    return round(acc*100,3)

In [62]:
X_train

,Pclass,Age,SibSp,Parch,Fare,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S
683,3,14.0,5,2,46.9000,0,1,0,0,1
320,3,22.0,0,0,7.2500,0,1,0,0,1
321,3,27.0,0,0,7.8958,0,1,0,0,1
450,2,36.0,1,2,27.7500,0,1,0,0,1
187,1,45.0,0,0,26.5500,0,1,0,0,1
...,...,...,...,...,...,...,...,...,...,...
227,3,20.5,0,0,7.2500,0,1,0,0,1
418,2,30.0,0,0,13.0000,0,1,0,0,1
195,1,58.0,0,0,146.5208,1,0,1,0,0
658,2,23.0,0,0,13.0000,0,1,0,0,1


In [63]:
# train_X['Embarked'].fillna('S', inplace=True)

y_train

683    0
320    0
321    0
450    0
187    1
      ..
227    0
418    0
195    1
658    0
230    1
Name: Survived, Length: 623, dtype: int64

In [66]:
model = RandomForestClassifier()
model.fit(X_train,y_train)
acc =evaluate(model)
print(acc)

77.654


In [74]:
model = LogisticRegression()
model.fit(X_train, y_train)
acc = evaluate(model)
print(f'LogisticRegression (accuracy): {acc}%')

model = DecisionTreeClassifier(criterion='gini', max_depth=12, random_state=42)
model.fit(X_train, y_train)
acc = evaluate(model)
print(f'DecisionTreeClassifier with gini (accuracy): {acc}%')

model = DecisionTreeClassifier(criterion='entropy', max_depth=12, random_state=42)
model.fit(X_train, y_train)
acc = evaluate(model)
print(f'DecisionTreeClassifier with entropy (accuracy): {acc}%')

model = RandomForestClassifier(n_estimators=100)
model.fit(X_train, y_train)
acc = evaluate(model)
# filename = 'model/RandomForestClassifier.sav'
# pickle.dump(model, open(filename, 'wb'))
# print(f'RandomForestClassifier (accuracy): {acc}%')

model = KNeighborsClassifier()
model.fit(X_train, y_train)
acc = evaluate(model)
print(f'KNeighborsClassifier (accuracy): {acc}%')

model = AdaBoostClassifier(n_estimators=100)
model.fit(X_train, y_train)
acc = evaluate(model)
print(f'AdaBoostClassifier (accuracy): {acc}%')

model = GradientBoostingClassifier(n_estimators=100)
model.fit(X_train, y_train)
acc = evaluate(model)
print(f'GradientBoostingClassifier (accuracy): {acc}%')

model = GaussianNB()
model.fit(X_train, y_train)
acc = evaluate(model)
print(f'GaussianNB (accuracy): {acc}%')

model = svm.SVC(kernel='rbf')
model.fit(X_train, y_train)
acc = evaluate(model)
print(f'SVC (accuracy): {acc}%')


/opt/conda/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression (accuracy): 78.731%
DecisionTreeClassifier with gini (accuracy): 79.478%
DecisionTreeClassifier with entropy (accuracy): 77.612%
KNeighborsClassifier (accuracy): 69.403%
AdaBoostClassifier (accuracy): 78.731%
GradientBoostingClassifier (accuracy): 82.463%
GaussianNB (accuracy): 77.239%
SVC (accuracy): 65.672%


In [73]:
from pandas import Series

feature_importance = model.feature_importances_
Series_feat_imp = Series(feature_importance, index=data.columns)

AttributeError: 'SVC' object has no attribute 'feature_importances_'

In [ ]:
plt.figure(figsize=(8,8))
Series_feat_imp.sort_values(ascending=True).plot.barh()
plt.xlabel('Feature Importance')
plt.ylabel('Feature')
plt.show()

In [91]:
test = pd.read_csv('/kaggle/input/titanic/test.csv')
submission = pd.read_csv('/kaggle/input/titanic/gender_submission.csv')

test['Embarked'].fillna('S', inplace=True)
test.loc[test.Fare.isnull(),'Fare'] = test['Fare'].mean()
test.loc[test.Age.isnull(),'Age'] = test['Age'].mean()

# labels = data["Survived"]
test_features = test.drop(columns=['PassengerId','PassengerId','Cabin','Ticket','Name'])
hot =pd.get_dummies(test_features,columns=['Sex','Embarked'])

test_features =hot
test_features

,Pclass,Age,SibSp,Parch,Fare,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S
0,3,34.50000,0,0,7.8292,0,1,0,1,0
1,3,47.00000,1,0,7.0000,1,0,0,0,1
2,2,62.00000,0,0,9.6875,0,1,0,1,0
3,3,27.00000,0,0,8.6625,0,1,0,0,1
4,3,22.00000,1,1,12.2875,1,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...
413,3,30.27259,0,0,8.0500,0,1,0,0,1
414,1,39.00000,0,0,108.9000,1,0,1,0,0
415,3,38.50000,0,0,7.2500,0,1,0,0,1
416,3,30.27259,0,0,8.0500,0,1,0,0,1


In [92]:
model = GradientBoostingClassifier(n_estimators=200)
model.fit(features,labels)
acc = evaluate(model)
print(f'GradientBoostingClassifier (accuracy): {acc}%')
prediction = model.predict(test_features)
submission['Survived'] = prediction
submission.to_csv('submission.csv', index=False)

GradientBoostingClassifier (accuracy): 90.299%
